# Breakdown Harness Engineering in Claude Code

## Initialization and Configuration

In [6]:
import os
import subprocess

from anthropic import Anthropic
from dotenv import load_dotenv

In [7]:
"""
Get Environment Variable
1. ANTHROPIC_API_KEY
2. ANTHROPIC_BASE_URL
3. ANTHROPIC_MODEL_ID
"""

load_dotenv(override=True)

client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url=os.getenv("ANTHROPIC_BASE_URL"),
)
MODEL = os.environ["ANTHROPIC_MODEL_ID"]
MAX_TOKEN = 8000

SYSTEM = f"You are a coding agent at {os.getcwd()}. Use bash to solve tasks. Act, don't explain."

TOOLS = [{
    "name": "bash",
    "description": "Run a shell command",
    "input_schema": {
        "type": "object",
        "properties": {
            "command": 
                {"type": "string"}
            },
        "required": ["command"],
    }
}]

## S01 Agent Tools

```text
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  Tool   |
| prompt |      |       |      | execute |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                    (loop until stop_reason != "tool_use")
```

一个退出条件控制整个流程。循环持续运行, 直到模型不再调用工具。

In [8]:
def run_bash(command: str) -> str:
    """
    The tool for agents to run bash
    """
    # Risky commands
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command will be blocked"
    try:
        r = subprocess.run(command,
                           shell=True,
                           cwd=os.getcwd(),
                           capture_output=True,
                           text=True,
                           timeout=120
                           )
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout(120s)"

In [9]:
def agent_loop(messages):
    """
    The main agent loop with bash
    """
    while True:
        response = client.messages.create(
            model=MODEL,
            system=SYSTEM,
            messages=messages,
            tools=TOOLS,
            max_tokens=MAX_TOKEN
        )
        messages.append({
            "role": "assistant",
            "content": response.content
        })

        # If LLM don't call any tool, we're done.
        if response.stop_reason != "tool_use":
            return

        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = run_bash(block.input["command"])
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                })

        messages.append({
            "role": "user",
            "content": results
        })

In [10]:
if __name__ == "__main__":
    """
    Main Process
    """
    history = []
    while True:
        try:
            query = input("\033[36ms01 >> \033[0m")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() in ("q", "exit", ""):
            break
        history.append({
            "role": "user",
            "content": query
        })
        agent_loop(history)
        response_content = history[-1]["content"]
        if isinstance(response_content, list):
            for block in response_content:
                if hasattr(block, "text"):
                    print(block.text)
        print()

Hello! I'm ready to help you with coding tasks. I'm currently in the `/Users/rong/Workspaces/2-Areas/21-Claude-Code/210-Harness-Agents` directory and can execute bash commands to assist you.

What would you like me to do?

There is 1 Python file in this directory:
- `main.py`

